# A팀 이상 데이터 증강 · 1~3단계
Colab에서 T4 GPU 런타임을 선택합니다. 고정 분할 검사 → 더미 스모크 → 복사-붙여넣기 → cTransGAN 학습·생성·평가 순서입니다.
실측 요약은 `results/baseline_summary.csv`, `results/ctransgan_t4_summary.csv`에 포함되어 있습니다.


In [ ]:
import subprocess
r = subprocess.run(['bash','-lc','nvidia-smi'],capture_output=True,text=True)
print(r.stdout or r.stderr)
# 1단계는 CPU로 실행 가능. 이후 학습 단계에서 GPU 필수 검사 예정.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/DL_TP')
PROJECT = DRIVE / 'runs/github_stage3/DL_TP'
if not (PROJECT / 'src/data/prepare.py').is_file():
    repo = Path('/content/26_2_DeepLearning')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/Junho-YEO-maker/26_2_DeepLearning.git', str(repo)], check=True)
    source = repo / 'Augmentation/visa_pcb'
    assert (source / 'splits/manifest.json').is_file()
    shutil.copytree(source, PROJECT, dirs_exist_ok=True)
os.chdir(PROJECT)


In [ ]:
%pip install -q -r requirements-stage2.txt

## 더미 데이터로 먼저 검증
라이브러리 설치 후 이미 numpy 등을 import했다면 런타임을 다시 시작하고 이 셀 전까지 재실행하세요. 실패하면 다음 단계로 넘어가지 않습니다.

In [ ]:
import subprocess,sys,json,yaml
from pathlib import Path
def run_logged(args, name):
    result=subprocess.run([sys.executable,*args],capture_output=True,text=True)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    (Path('reports')/name).write_text(result.stdout)
run_logged(['-m','tests.smoke_stage1'],'colab_smoke_stage1.json')

## 원본 다운로드 및 PCB만 추출
약 1.93 GB tar 파일과 PCB 원본이 Drive에 저장됩니다. 다운로드가 끊겼으면 재실행하면 이어받습니다. 이미지와 마스크는 리사이즈하지 않습니다.

In [ ]:
config = yaml.safe_load(Path('configs/data.yaml').read_text())
ROOT = str(DRIVE / 'data/VisA_stage3_20261111')
provenance = json.loads(Path('reports/archive_provenance.json').read_text())
if not all((Path(ROOT) / pcb / 'image_anno.csv').is_file() for pcb in config['pcb_types']):
    subprocess.run([sys.executable, '-m', 'src.data.download', '--destination', ROOT,
                    '--expected-sha', provenance['sha256']], check=True)


## 고정 분할 확인
seed=7017, 불량 33/67, 정상 80/20. 제공된 고정 CSV와 manifest를 원본·마스크 해시와 교차 검증합니다.


In [ ]:
run_logged(['-m','src.data.audit','--root',ROOT,'--splits','splits'],'colab_real_audit.json')


## 고정 분할 요약

In [ ]:
manifest=json.loads(Path('splits/manifest.json').read_text())
for pcb,info in manifest['pcb'].items():
    print(pcb,info['counts'],info['class_counts'])
print('1단계 검증 완료')

## 단계 2 · 더미 테스트, 생성, 평가
500장씩 생성합니다. 결과 ZIP을 먼저 풀었다면 완료 레코드와 해시를 확인해 재사용합니다.

In [ ]:
os.environ['TORCH_HOME']=str(PROJECT/'cache/torch')
run_logged(['-m','tests.smoke_stage2'],'colab_smoke_stage2.log')

In [ ]:
run_logged(['-m','src.generate.run','--data-root',ROOT],'colab_generation_stage2.log')

In [ ]:
run_logged(['-m','src.metrics.evaluate','--data-root',ROOT],'colab_evaluation_stage2.log')
run_logged(['-m','src.metrics.summarize'],'colab_summary_stage2.log')

In [ ]:
from IPython.display import display, Image
for pcb in ['pcb1','pcb2','pcb3','pcb4']:
    display(Image(filename=f'figures/{pcb}_closeups.png'))

## 3단계 · cTransGAN 구현 검증
논문의 클래스 특징 임베딩과 패치 생성·재삽입 흐름을 축소 구현했습니다. STAGE3_CTRANSGAN.md에 원논문 대비 변경점과 픽셀 마스크 해석을 적었습니다. 먼저 더미 데이터 smoke test를 실행합니다.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['TORCH_HOME']=str(PROJECT/'cache/torch')
def run_streamed(arguments, log_name):
    with (PROJECT/'reports'/log_name).open('w') as log:
        with subprocess.Popen([sys.executable,*arguments],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
            if process.wait()!=0:
                raise RuntimeError(f'실행 실패. reports/{log_name}을 확인하세요.')
run_streamed(['-m','tests.smoke_stage3'],'colab_smoke_stage3.log')

## GPU 확인과 프로필 선택
이 셀에서 GPU가 확인되어야 본학습을 시작합니다. T4 프로필은 micro batch 8, gradient accumulation 2, fp16, gradient checkpointing을 사용합니다. A100은 별도 YAML을 선택합니다.

In [ ]:
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(), 'Colab 런타임을 GPU로 변경하세요.'
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME)
profile = 'ctransgan_a100' if 'A100' in GPU_NAME else 'ctransgan_t4'
settings = yaml.safe_load(Path(f'configs/{profile}.yaml').read_text())
settings['data_root'] = ROOT
GAN_CONFIG = f'configs/{profile}_runtime.yaml'
Path(GAN_CONFIG).write_text(yaml.safe_dump(settings, sort_keys=False))
print(Path(GAN_CONFIG).read_text())


## PCB별 분류기 → 클래스 임베딩 → GAN 학습
분류기 200 업데이트와 GAN 4,000 업데이트를 사용합니다. 100 업데이트마다 모델·옵티마이저·AMP scaler·난수 상태를 저장합니다. 같은 셀을 다시 실행하면 마지막 완료 체크포인트에서 재개합니다. 시간·GPU 피크 메모리와 실제 설정은 checkpoints/ctransgan*/{pcb}/에 기록됩니다.

In [ ]:
run_streamed(['-m','src.train.ctransgan','--config',GAN_CONFIG],'colab_ctransgan_train.log')

## 기본 500장씩 생성 및 동일 지표 평가
학습 계획이 완료된 체크포인트만 사용합니다. 출력 마스크는 삽입 영역이며 생성 내용의 의미적 결함 정확도는 정성 평가와 B팀 평가에서 확인해야 합니다.

In [ ]:
run_streamed(['-m','src.generate.run','--config',GAN_CONFIG],'colab_ctransgan_generate.log')
run_streamed(['-m','src.metrics.evaluate','--config',GAN_CONFIG],'colab_ctransgan_evaluate.log')
run_streamed(['-m','src.metrics.stage3_summary'],'colab_stage3_summary.log')
from IPython.display import display,Image
for pcb in ['pcb1','pcb2','pcb3','pcb4']:
    display(Image(filename=f'figures/ctransgan/{pcb}_closeups.png'))

3단계 결과는 results/ctransgan/ 및 figures/ctransgan/, 비교표는 results/stage3_comparison.csv입니다. 단계 결과를 확인한 뒤 다음 모델로 진행합니다.